In [38]:
import numpy as np
from sklearn.datasets import load_iris

iris = load_iris()

# IRIS feature names:
feature_names = iris.feature_names

# IRIS feature values (cm) in numpy ndarray:
data = iris.data

# IRIS species names
species_names = iris.target_names

# IRIS species ids (0='setosa', 1='versicolor', 2='virginica'):
species = iris.target

# Training and test data

X = data
y_setosa = (species == 0).astype(int)  # 转为二分类：setosa(0)为1，其它为0
y_versicolor = (species == 1).astype(int)  # 转为二分类：versicolor(1)为1，其它为0
y_virginica = (species == 2).astype(int)  # 转为二分类：virginica(2)为1，其它为0

# 训练集和测试集划分

In [39]:
rng = np.random.default_rng(seed=42) 
train_idx, val_idx, test_idx = [],[],[]
for i in range(3):
    idx = np.where(species == i)[0]  # 获取当前类别的索引
    idx = rng.permutation(idx)  # 打乱索引
    n = len(idx)
    train_idx.extend(idx[:int(0.4*n)])  # 前40%作为训练集
    val_idx.extend(idx[int(0.4*n):int(0.7*n)])  # 中间30%作为验证集
    test_idx.extend(idx[int(0.7*n):])  # 后30%作为测试集
train_idx = np.array(train_idx)
val_idx = np.array(val_idx)
test_idx = np.array(test_idx)
X_train, y_setosa_train, y_versicolor_train, y_virginica_train = X[train_idx], y_setosa[train_idx], y_versicolor[train_idx], y_virginica[train_idx]
X_val, y_setosa_val, y_versicolor_val, y_virginica_val = X[val_idx], y_setosa[val_idx], y_versicolor[val_idx], y_virginica[val_idx]
X_test, y_setosa_test, y_versicolor_test, y_virginica_test = X[test_idx], y_setosa[test_idx], y_versicolor[test_idx], y_virginica[test_idx]



In [40]:
class LogisticRegression:
    def __init__(self, in_dim, out_dim, reg='l1norm', alpha=0.):
        rng = np.random.default_rng(seed=0)
        self.weight = rng.normal(loc=0, scale=1, size=in_dim)
        #self.weight = rng.normal(loc=0, scale=1, size=(in_dim, out_dim))#这里的炸裂操作把数组变为了二维的了
        self.bias = rng.normal(loc=0, scale=1, size=out_dim)
        self.reg = reg
        self.alpha = alpha

    def sigmoid(self, x):
        x = np.clip(x, -500, 500)
        return 1 / (1 + np.exp(-x))

    def forward(self, x):
        x = np.dot(x, self.weight) + self.bias
        x = self.sigmoid(x)
        return x
    
    def compute_reg_loss(self):
        if self.reg == 'l1norm':
            reg_loss = self.alpha * np.sum(np.abs(self.weight))
        return reg_loss

    def compute_reg_grad(self):
        if self.reg == 'l1norm':
            reg_grad = self.alpha * np.sign(self.weight)
        return reg_grad

    def compute_loss(self, y_true, y_prob):
        m = len(y_true)
        epsilon = 1e-15
        y_prob = np.clip(y_prob, epsilon, 1 - epsilon) # avoid log(0)
        loss = -1/m * np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
        penalty = self.compute_reg_loss()
        return loss + penalty

    def compute_gradient(self, x, y_true, y_prob):
        m = len(y_true)
        dw = (1/m) * np.dot(x.T, (y_prob - y_true))
        dw += self.compute_reg_grad()
        db = (1/m) * np.sum(y_prob - y_true)
        return dw, db

    def step_update(self, dw, db, learning_rate=0.01):
        self.weight -= learning_rate * dw
        self.bias -= learning_rate * db

In [41]:
batch_size = 16

# 验证集上调整learning rate和正则化强度alpha
def train_and_eval(y_train, y_val, lr, alpha, epochs=5000):
    model = LogisticRegression(in_dim=X_train.shape[1], out_dim=1, reg='l1norm', alpha=alpha)
    for epoch in range(epochs):
        rng_1 = np.random.default_rng(seed=140) 
        idx = rng_1.permutation(len(X_train))
        X_shuffled = X_train[idx]
        y_shuffled = y_train[idx]
        for i in range(0, len(X_train), batch_size):
            X_batch = X_shuffled[i:i + batch_size]
            y_batch = y_shuffled[i:i + batch_size]
            y_prob = model.forward(X_batch)
            dw, db = model.compute_gradient(X_batch, y_batch, y_prob)
            model.step_update(dw, db, learning_rate=lr)
    val_loss = model.compute_loss(y_val, model.forward(X_val))
    return val_loss, model

lr_grid = [1e-3, 1e-2, 1e-1]
alpha_grid = [0.0, 1e-3, 1e-2, 1e-1]

best_models = {}
best_params = {}

for class_name, y_train, y_val in [
    ("setosa", y_setosa_train, y_setosa_val),
    ("versicolor", y_versicolor_train, y_versicolor_val),
    ("virginica", y_virginica_train, y_virginica_val),
]:
    best_loss = np.inf
    for lr in lr_grid:
        for alpha in alpha_grid:
            val_loss, model = train_and_eval(y_train, y_val, lr, alpha)
            print(f"{class_name} | lr={lr:.3g}, alpha={alpha:.3g}, val_loss={val_loss:.4f}")
            if val_loss < best_loss:
                best_loss = val_loss
                best_models[class_name] = model
                best_params[class_name] = {"lr": lr, "alpha": alpha} #字典的元素也是字典，包含了lr和alpha两个键值对
    print(f"{class_name}: best val loss={best_loss:.4f}, params={best_params[class_name]}")

# 测试集上评估三个模型的性能
for class_name, model in best_models.items():
    if class_name == "setosa":
        y_test = y_setosa_test
    elif class_name == "versicolor":
        y_test = y_versicolor_test
    else:
        y_test = y_virginica_test
    test_loss = model.compute_loss(y_test, model.forward(X_test))
    print(f"{class_name} test loss: {test_loss:.4f}")



setosa | lr=0.001, alpha=0, val_loss=0.0335
setosa | lr=0.001, alpha=0.001, val_loss=0.0386
setosa | lr=0.001, alpha=0.01, val_loss=0.0813
setosa | lr=0.001, alpha=0.1, val_loss=0.3217
setosa | lr=0.01, alpha=0, val_loss=0.0042
setosa | lr=0.01, alpha=0.001, val_loss=0.0125
setosa | lr=0.01, alpha=0.01, val_loss=0.0651
setosa | lr=0.01, alpha=0.1, val_loss=0.2498
setosa | lr=0.1, alpha=0, val_loss=0.0006
setosa | lr=0.1, alpha=0.001, val_loss=0.0106
setosa | lr=0.1, alpha=0.01, val_loss=0.0550
setosa | lr=0.1, alpha=0.1, val_loss=0.2456
setosa: best val loss=0.0006, params={'lr': 0.1, 'alpha': 0.0}
versicolor | lr=0.001, alpha=0, val_loss=0.5693
versicolor | lr=0.001, alpha=0.001, val_loss=0.5712
versicolor | lr=0.001, alpha=0.01, val_loss=0.5852
versicolor | lr=0.001, alpha=0.1, val_loss=0.6347
versicolor | lr=0.01, alpha=0, val_loss=0.5199
versicolor | lr=0.01, alpha=0.001, val_loss=0.5260
versicolor | lr=0.01, alpha=0.01, val_loss=0.5606
versicolor | lr=0.01, alpha=0.1, val_loss=0.6